# Project: Knowledge distillation 🧪➡️🐣
Big models are accurate, small models are fast. **Knowledge distillation** (Hinton, Vinyals & Dean 2015) trains a small *student* to imitate a large *teacher*'s output distribution, not just the hard labels. You'll distill a CNN (≈ 420k parameters) into a tiny MLP (≈ 25k parameters, 17× smaller) on Fashion-MNIST, look at the teacher's "dark knowledge", sweep the temperature, and find out when distillation helps a little and when it helps a lot.

Topic: [[Model Deployment and Serving]] (model compression), [[Fine-tuning and Alignment]] (distillation for LLMs) · guide note: [[Project - Knowledge Distillation]] · paper: [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)

Runtime ≈ 1–2 minutes on a CPU (most of it is training the teacher). The Fashion-MNIST download is ≈ 30 MB.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, torchvision
torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
CLASSES = ["T-shirt", "trouser", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "ankle boot"]
tr = torchvision.datasets.FashionMNIST("data", train=True, download=True); te = torchvision.datasets.FashionMNIST("data", train=False, download=True)
X_all, y_all = tr.data.float().unsqueeze(1) / 255, tr.targets; X_test, y_test = te.data.float().unsqueeze(1) / 255, te.targets
X_lab, y_lab = X_all[:2000], y_all[:2000]          # the student only gets 2,000 labelled images…
X_unl = X_all[2000:32000]                           # …and (in part 4) 30,000 unlabelled ones

def fit(model, n, loss_fn, epochs, lr=1e-3, bs=128):
    opt = torch.optim.Adam(model.parameters(), lr)
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(n).split(bs):
            loss = loss_fn(model, idx); opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()
def predict(model, X):
    with torch.no_grad(): return torch.cat([model(X[i:i + 2500]) for i in range(0, len(X), 2500)])
def accuracy(model): return (predict(model, X_test).argmax(1) == y_test).float().mean().item()
def n_params(m): return sum(p.numel() for p in m.parameters())
make_student = lambda: nn.Sequential(nn.Flatten(), nn.Linear(784, 32), nn.ReLU(), nn.Linear(32, 10))

## 0. The teacher
A small CNN trained on all 60,000 labelled images (4 epochs, ≈ 1 minute). In real life this would be the big model you already have.

In [ ]:
torch.manual_seed(0); t0 = time.time()
teacher = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                        nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10))
teacher = fit(teacher, len(X_all), lambda m, i: F.cross_entropy(m(X_all[i]), y_all[i]), epochs=4, bs=256)
T_lab, T_unl = predict(teacher, X_lab), predict(teacher, X_unl)          # teacher logits, computed once
print(f"teacher: {n_params(teacher):,} parameters, test accuracy {accuracy(teacher):.1%} ({time.time() - t0:.0f}s)")
print(f"student: {n_params(make_student()):,} parameters ({n_params(teacher) / n_params(make_student()):.0f}× smaller)")

## 1. Dark knowledge
A hard label says "shirt". The teacher's softmax says something like *shirt 0.7, T-shirt 0.2, pullover 0.08, … sandal 0.0001*: which wrong answers are plausible. Dividing the logits by a **temperature** $T>1$ before the softmax flattens the distribution and makes those small probabilities visible.

In [ ]:
i = int((y_lab == 6).nonzero()[0])                                    # first shirt in the labelled set
fig, ax = plt.subplots(1, 4, figsize=(15, 2.8)); ax[0].imshow(X_lab[i, 0], cmap="gray"); ax[0].set_title("a shirt"); ax[0].axis("off")
for a, T in zip(ax[1:], (1, 4, 10)):
    p = F.softmax(T_lab[i] / T, 0).numpy(); a.bar(range(10), p); a.set_xticks(range(10)); a.set_xticklabels(CLASSES, rotation=60, fontsize=7); a.set_title(f"teacher, T = {T}")
plt.show()

## 2. The distillation loss
$$\mathcal L_{KD} = T^2\;\mathrm{KL}\big(\text{softmax}(z_t/T)\;\big\|\;\text{softmax}(z_s/T)\big)$$
Use `F.kl_div(log_probs_student, probs_teacher, reduction="batchmean")`, which expects the **student's log-probabilities** and the **teacher's probabilities**. The $T^2$ factor keeps gradient magnitudes comparable across temperatures (the soft gradients scale as $1/T^2$).

In [ ]:
def kd_loss(student_logits, teacher_logits, T):
    # TODO 1: T² · KL(teacher softened ‖ student softened), batch mean
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_kd():
    zs, zt = torch.tensor([[1.0, 2.0, 0.5]]), torch.tensor([[2.0, 0.0, 1.0]])
    pt, ps = F.softmax(zt / 2, 1), F.softmax(zs / 2, 1); manual = 4 * (pt * (pt.log() - ps.log())).sum()
    return abs(kd_loss(zs, zt, 2).item() - manual.item()) < 1e-6 and abs(kd_loss(zt, zt, 3).item()) < 1e-6
check("KD loss", _t_kd, "Must equal T²·Σ p_t (log p_t − log p_s) and be 0 when student = teacher.")

## 3. The full student loss
$\mathcal L = (1-\alpha)\,\mathrm{CE}(z_s, y) + \alpha\,\mathcal L_{KD}$: a mix of the true labels and the teacher's soft targets.

In [ ]:
def distill_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.7):
    # TODO 2: weighted sum of cross-entropy with the labels and kd_loss
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_total():
    zs, zt, y = torch.randn(4, 10), torch.randn(4, 10), torch.tensor([1, 2, 3, 4])
    return torch.isclose(distill_loss(zs, zt, y, 2, 0.0), F.cross_entropy(zs, y)) and torch.isclose(distill_loss(zs, zt, y, 2, 1.0), kd_loss(zs, zt, 2))
check("distill loss", _t_total, "alpha = 0 must give plain cross-entropy, alpha = 1 pure KD.")

## 4. Experiment A: same 2,000 labelled images
Train the student three times per setting (different seeds) on the 2,000 labelled images: hard labels only vs. distillation at several temperatures (α = 1: soft targets only; α = 0.7: a mix).

In [ ]:
results = {}
if ready("KD loss", "distill loss"):
    settings = {"hard labels": None, "KD T=1": (1, 1.0), "KD T=4, α=0.7": (4, 0.7), "KD T=8": (8, 1.0)}
    for name, cfg in settings.items():
        accs = []
        for seed in range(3):
            torch.manual_seed(seed)
            if cfg is None: lf = lambda m, i: F.cross_entropy(m(X_lab[i]), y_lab[i])
            else: lf = lambda m, i, T=cfg[0], a=cfg[1]: distill_loss(m(X_lab[i]), T_lab[i], y_lab[i], T, a)
            accs.append(accuracy(fit(make_student(), len(X_lab), lf, epochs=40)))
        results[name] = float(np.mean(accs)); print(f"{name:<16} test accuracy {np.round(accs, 3)} → mean {results[name]:.1%}")

**What you should see:** a modest gain at T = 1 and *worse* results at high temperatures. Hinton et al. noted that when the student is much smaller than the teacher, intermediate or low temperatures work best: a 25k-parameter MLP simply can't match the fine detail of very soft targets. Distillation is not magic on the same data.

## 5. Experiment B: the teacher labels data for you
Distillation's real superpower: the teacher can label **unlabelled** data. Train the student on the 2,000 labelled + 30,000 unlabelled images using only the teacher's soft targets, and compare with the upper bound of a student trained on 60,000 *true* labels.

In [ ]:
if ready("KD loss", "distill loss"):
    X_tr = torch.cat([X_lab, X_unl]); T_tr = torch.cat([T_lab, T_unl])
    torch.manual_seed(0); transfer = fit(make_student(), len(X_tr), lambda m, i: kd_loss(m(X_tr[i]), T_tr[i], 1.0), epochs=8)
    results["KD on 32k (teacher-labelled)"] = accuracy(transfer)
    torch.manual_seed(0); upper = fit(make_student(), len(X_all), lambda m, i: F.cross_entropy(m(X_all[i]), y_all[i]), epochs=8)
    results["upper bound: 60k true labels"] = accuracy(upper)
    for k in ("hard labels", "KD on 32k (teacher-labelled)", "upper bound: 60k true labels"): print(f"{k:<30} {results[k]:.1%}")
    pd_ = {k: v for k, v in results.items()}
    plt.barh(list(pd_), list(pd_.values())); plt.xlim(0.75, 0.9); plt.xlabel("test accuracy"); plt.title(f"student ({n_params(make_student()):,} params) vs teacher {accuracy(teacher):.1%}"); plt.show()
    check("distillation pays off", lambda: results["KD T=1"] > results["hard labels"] and results["KD on 32k (teacher-labelled)"] > results["hard labels"] + 0.03,
          "KD at T=1 should beat hard labels, and distilling on teacher-labelled unlabelled data should add ≥ 3 points.")

<details><summary>▶ Solution</summary>

```python
def kd_loss(student_logits, teacher_logits, T):
    log_p_s = F.log_softmax(student_logits / T, dim=1)
    p_t = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(log_p_s, p_t, reduction="batchmean") * T * T
```

```python
def distill_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.7):
    hard = F.cross_entropy(student_logits, labels)
    return (1 - alpha) * hard + alpha * kd_loss(student_logits, teacher_logits, T)
```
</details>

## Stretch goals
- **Bigger student** (784 → 256 → 10): does the best temperature move up? (It should, following Hinton et al.)
- **Measure speed:** time 10,000 predictions for teacher and student. Is the 17× parameter reduction a 17× speed-up?
- **Quantise** the student to int8 (`torch.ao.quantization.quantize_dynamic`) and combine both compressions ([[Model Deployment and Serving]]).
- **LLM version:** distillation for language models = fine-tuning a small model on the big model's generated answers ([[Fine-tuning and Alignment]]). Distil the [[Project - Build a Transformer from Scratch]] into a 1-layer version.